<table>
    <tr>
        <td colspan=2><strong>International University of La Rioja (UNIR)</strong></td>
    </tr>
    <tr>
        <td colspan=2><strong>Master's Degree in Mathematical Engineering and Computing</strong></td>
    </tr>
    <tr>
        <td>Master's thesis submitted by:</td>
        <td><strong>Andrés Benavides, Diego Salamanca, Yeferson Salamanca.</strong></td>
    </tr>
    <tr>
        <td>Supervisor:</td>
        <td>PhD. Juan Francisco Martínez Cerda</td>
    </tr>
</table>


---


# 01 - Download financial data from Yahoo Finance

This notebook downloads historical OHLCV data for the 25 sector tickers plus 3 macro tickers (`^VIX`, `DX-Y.NYB`, `^TNX`) defined in `metadata/tickers.json`, with stocks distributed across 5 sectors (technology, healthcare, agriculture, basic materials, financial services).

**Inputs**
- `metadata/tickers.json` - sector -> list of tickers, plus the default `period` (12 years) and `interval` (`1d`).

**Outputs**
- One CSV per ticker in `data/raw/<TICKER>.csv`.
- A download report saved in `data/raw/_download_report.json`.

**Behavior**
- Each ticker is downloaded with up to 3 attempts, waiting ~20 seconds between attempts (so an individual ticker can recover from a transient error in under a minute).
- A final report prints the number of successfully downloaded tickers and the list of failures.
- The last cell allows any individual ticker to be downloaded again manually.

Merging the tickers into a single dataset is intentionally outside the scope here and will be covered in a later notebook.

## 1. Imports and paths

In [1]:
import json
import logging
import time
from pathlib import Path

import pandas as pd
import yfinance as yf

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("download_data")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METADATA_PATH = PROJECT_ROOT / "metadata" / "tickers.json"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_PATH = RAW_DIR / "_download_report.json"

METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"METADATA_PATH : {METADATA_PATH}")
print(f"RAW_DIR       : {RAW_DIR}")

PROJECT_ROOT  : /Applications/TFM_LSTM
METADATA_PATH : /Applications/TFM_LSTM/metadata/tickers.json
RAW_DIR       : /Applications/TFM_LSTM/data/raw


## 2. Load (or initialize) ticker metadata

Load `metadata/tickers.json` and flatten it into a list of `(sector, ticker)` pairs that the rest of the notebook iterates over.

In [ ]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing metadata file: {METADATA_PATH}. Run notebook 01 first."
    )

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

PERIOD = f"{metadata.get('default_period_years')}y"
INTERVAL = metadata.get("default_interval")

print(f"Period          : {PERIOD}")
print(f"Interval        : {INTERVAL}")

Period          : 12y
Interval        : 1d


In [4]:
ticker_pairs = [
    (sector, ticker)
    for sector, tickers in metadata["sectors"].items()
    for ticker in tickers.keys()
]

print(f"Sectors         : {list(metadata['sectors'].keys())}")
print(f"Total tickers   : {len(ticker_pairs)}")
ticker_pairs

Sectors         : ['technology', 'healthcare', 'agriculture', 'energy', 'financial_services']
Total tickers   : 25


[('technology', 'AAPL'),
 ('technology', 'MSFT'),
 ('technology', 'NVDA'),
 ('technology', 'GOOGL'),
 ('technology', 'AMZN'),
 ('healthcare', 'LLY'),
 ('healthcare', 'UNH'),
 ('healthcare', 'JNJ'),
 ('healthcare', 'MRK'),
 ('healthcare', 'ISRG'),
 ('agriculture', 'DE'),
 ('agriculture', 'ADM'),
 ('agriculture', 'BG'),
 ('agriculture', 'MOS'),
 ('agriculture', 'CF'),
 ('energy', 'XOM'),
 ('energy', 'CVX'),
 ('energy', 'COP'),
 ('energy', 'EOG'),
 ('energy', 'SLB'),
 ('financial_services', 'JPM'),
 ('financial_services', 'BLK'),
 ('financial_services', 'GS'),
 ('financial_services', 'MS'),
 ('financial_services', 'AXP')]

## 3. Download helper with retries

`download_ticker` attempts to retrieve a single ticker up to `max_attempts` times. It waits `wait_seconds` between attempts. With the defaults (3 attempts, 20 s wait), the worst-case total waiting time per ticker remains below one minute, satisfying the "retry in under a minute" requirement.

A response is considered a failure when `yfinance` raises an exception or returns an empty DataFrame (Yahoo sometimes does this during transient issues).

In [ ]:
def download_ticker(
    ticker: str,
    period: str = PERIOD,
    interval: str = INTERVAL,
    max_attempts: int = 3,
    wait_seconds: int = 20,
) -> pd.DataFrame:
    """Download OHLCV data for a single ticker, retrying on errors / empty results."""
    last_error: Exception | None = None
    for attempt in range(1, max_attempts + 1):
        try:
            logger.info("[%s] attempt %d/%d (period=%s, interval=%s)",
                        ticker, attempt, max_attempts, period, interval)
            df = yf.download(ticker,period=period,interval=interval,auto_adjust=False,progress=False,threads=False)
            if df is None or df.empty:
                raise RuntimeError("yfinance returned an empty DataFrame")
            logger.info("[%s] downloaded %d rows", ticker, len(df))
            return df
        except Exception as exc:
            last_error = exc
            logger.warning("[%s] attempt %d failed: %s", ticker, attempt, exc)
            if attempt < max_attempts:
                logger.info("[%s] sleeping %ds before retry", ticker, wait_seconds)
                time.sleep(wait_seconds)

    raise RuntimeError(
        f"Failed to download {ticker} after {max_attempts} attempts: {last_error}"
    )

## 4. Save helper

Each ticker is saved in `data/raw/<TICKER>.csv`, including the date index. The ticker's filename is used as-is (e.g. `YAR.OL.csv`).

In [6]:
def save_ticker_csv(df: pd.DataFrame, ticker: str, out_dir: Path = RAW_DIR) -> Path:
    """Persist a ticker DataFrame as <TICKER>.csv and return the output path."""
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{ticker}.csv"
    df.to_csv(out_path, index=True)
    logger.info("[%s] saved %s (%d rows)", ticker, out_path, len(df))
    return out_path

## 5. Batch download of all tickers

Iterate over each `(sector, ticker)` pair, download with retries, save the CSV, and record success or failure. Failures are set aside so the loop continues with the remaining tickers.

In [7]:
successes: list[dict] = []
failures: list[dict] = []

In [8]:
for sector, ticker in ticker_pairs:
    logger.info("--- %s / %s ---", sector, ticker)
    try:
        df = download_ticker(ticker)
        out_path = save_ticker_csv(df, ticker)
        successes.append({
            "sector": sector,
            "ticker": ticker,
            "rows": int(len(df)),
            "path": str(out_path),
        })
    except Exception as exc:
        logger.error("[%s] giving up: %s", ticker, exc)
        failures.append({
            "sector": sector,
            "ticker": ticker,
            "error": str(exc),
        })

print(f"\nDone. Success: {len(successes)} | Failures: {len(failures)}")

08:28:26 | INFO | --- technology / AAPL ---
08:28:26 | INFO | [AAPL] attempt 1/3 (period=12y, interval=1d)
08:28:27 | INFO | [AAPL] downloaded 3018 rows
08:28:27 | INFO | [AAPL] saved /Applications/TFM_LSTM/data/raw/AAPL.csv (3018 rows)
08:28:27 | INFO | --- technology / MSFT ---
08:28:27 | INFO | [MSFT] attempt 1/3 (period=12y, interval=1d)
08:28:27 | INFO | [MSFT] downloaded 3018 rows
08:28:27 | INFO | [MSFT] saved /Applications/TFM_LSTM/data/raw/MSFT.csv (3018 rows)
08:28:27 | INFO | --- technology / NVDA ---
08:28:27 | INFO | [NVDA] attempt 1/3 (period=12y, interval=1d)
08:28:28 | INFO | [NVDA] downloaded 3018 rows
08:28:28 | INFO | [NVDA] saved /Applications/TFM_LSTM/data/raw/NVDA.csv (3018 rows)
08:28:28 | INFO | --- technology / GOOGL ---
08:28:28 | INFO | [GOOGL] attempt 1/3 (period=12y, interval=1d)
08:28:28 | INFO | [GOOGL] downloaded 3018 rows
08:28:28 | INFO | [GOOGL] saved /Applications/TFM_LSTM/data/raw/GOOGL.csv (3018 rows)
08:28:28 | INFO | --- technology / AMZN ---
08:


Done. Success: 25 | Failures: 0


## 5b. Batch download of macro tickers

The same retry/save logic as the sector loop, but iterating over the macro tickers defined under the top-level `macro` key in `metadata/tickers.json` (`^VIX`, `DX-Y.NYB`, `^TNX`).

Successes and failures are appended to the same `successes` / `failures` lists used in section 5, so the download report in section 6 covers everything.

In [9]:
macro_tickers = list(metadata.get("macro", {}).keys())
print(f"Macro tickers   : {macro_tickers}")

for ticker in macro_tickers:
    logger.info("--- macro / %s ---", ticker)
    try:
        df = download_ticker(ticker)
        out_path = save_ticker_csv(df, ticker)
        successes.append({
            "sector": "macro",
            "ticker": ticker,
            "rows": int(len(df)),
            "path": str(out_path),
        })
    except Exception as exc:
        logger.error("[%s] giving up: %s", ticker, exc)
        failures.append({
            "sector": "macro",
            "ticker": ticker,
            "error": str(exc),
        })

print(f"\nMacro done. Successes so far: {len(successes)} | Failures: {len(failures)}")

08:28:36 | INFO | --- macro / ^VIX ---
08:28:36 | INFO | [^VIX] attempt 1/3 (period=12y, interval=1d)


Macro tickers   : ['^VIX', 'DX-Y.NYB', '^TNX']


08:28:37 | INFO | [^VIX] downloaded 3020 rows
08:28:37 | INFO | [^VIX] saved /Applications/TFM_LSTM/data/raw/^VIX.csv (3020 rows)
08:28:37 | INFO | --- macro / DX-Y.NYB ---
08:28:37 | INFO | [DX-Y.NYB] attempt 1/3 (period=12y, interval=1d)
08:28:37 | INFO | [DX-Y.NYB] downloaded 3019 rows
08:28:37 | INFO | [DX-Y.NYB] saved /Applications/TFM_LSTM/data/raw/DX-Y.NYB.csv (3019 rows)
08:28:37 | INFO | --- macro / ^TNX ---
08:28:37 | INFO | [^TNX] attempt 1/3 (period=12y, interval=1d)
08:28:38 | INFO | [^TNX] downloaded 3018 rows
08:28:38 | INFO | [^TNX] saved /Applications/TFM_LSTM/data/raw/^TNX.csv (3018 rows)



Macro done. Successes so far: 28 | Failures: 0


## 6. Download report

Print a summary and save the report in `data/raw/_download_report.json` for later inspection.

In [10]:
total = len(ticker_pairs)
n_ok = len(successes)
n_fail = len(failures)

print("=" * 60)
print("DOWNLOAD REPORT")
print("=" * 60)
print(f"Total tickers attempted : {total}")
print(f"Successful downloads    : {n_ok}")
print(f"Failed downloads        : {n_fail}")
print("-" * 60)

if failures:
    print("Failures:")
    failures_df = pd.DataFrame(failures)
    print(failures_df.to_string(index=False))
else:
    print("No failures.")

report = {
    "period": PERIOD,
    "interval": INTERVAL,
    "total": total,
    "successful": n_ok,
    "failed": n_fail,
    "successes": successes,
    "failures": failures,
}
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("-" * 60)
print(f"Report saved to: {REPORT_PATH}")

DOWNLOAD REPORT
Total tickers attempted : 25
Successful downloads    : 28
Failed downloads        : 0
------------------------------------------------------------
No failures.
------------------------------------------------------------
Report saved to: /Applications/TFM_LSTM/data/raw/_download_report.json


## 7. Manual download of a single ticker

Edit the following three variables to download a symbol again (or retrieve a new one) on demand. This uses the same `download_ticker` + `save_ticker_csv` helpers as the batch loop, so the CSV ends up in `data/raw/` and is consistent with the rest of the dataset.

In [13]:
# Change this to True to run the cell, download a single ticker and save it to data/raw/ manually.
run_cell = False

if run_cell:
    MANUAL_TICKER = "JCI"
    MANUAL_PERIOD = "1y"
    MANUAL_INTERVAL = "1d"

    manual_df = download_ticker(
        MANUAL_TICKER,
        period=MANUAL_PERIOD,
        interval=MANUAL_INTERVAL,
        max_attempts=1
    )
    manual_path = save_ticker_csv(manual_df, MANUAL_TICKER)

    print(f"Manually downloaded {MANUAL_TICKER}: {len(manual_df)} rows -> {manual_path}")
    manual_df.head()